# Titli NIDS Testing Notebook

This notebook demonstrates the complete workflow of the Titli Network Intrusion Detection System:
1. Install Titli package
2. Feature extraction from PCAP files
3. Model training
4. Model saving and loading
5. Inference
6. Evaluation

## Step 1: Install Titli Package

Install the titli package in editable mode from the parent directory.

In [ ]:
# Install titli package
!pip install -e ../titli

## Step 2: Import Required Libraries

Import all necessary modules from titli and other dependencies.

In [ ]:
from titli.fe import AfterImage, NetStat
from scapy.all import *

from titli.utils import StreamingCSVDataset
from torch.utils.data import DataLoader
import torch
from titli.ids import VAE, Autoencoder, KitNET, ICL, LOF, OCSVM

print("All imports successful!")

## Step 3: Configure Paths and Device

Set up the file paths for PCAP data and configure the compute device (CPU/GPU).

In [ ]:
# Configure paths
PCAP_PATH = "./utils/iot_datasets/uq-iot/benign/weekday_100k.pcap"
FEATURE_PATH = "./utils/iot_datasets/uq-iot/benign/weekday_100k1.csv"
LABEL_PATH = "./single_col_zeros.csv"

# Configure device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset_name = "uq-iot"

print(f"Using device: {device}")
print(f"PCAP path: {PCAP_PATH}")
print(f"Feature output: {FEATURE_PATH}")

## Step 4: Feature Extraction

Extract features from the PCAP file using the AfterImage feature extractor.

In [ ]:
# Initialize feature extractor
fe = AfterImage(file_path=PCAP_PATH)

# Extract features and save to CSV
fe.extract_features(output_path=FEATURE_PATH)

print(f"Feature extraction complete! Output saved to {FEATURE_PATH}")

## Step 5: Create Training DataLoader

Set up a streaming dataset and DataLoader for efficient training.

In [ ]:
# Create streaming dataset
streaming_dataset = StreamingCSVDataset(
    feature_csv_path=FEATURE_PATH,
    label_csv_path=LABEL_PATH,
    max_samples=100000,  # Limit to 100k samples
    label_column=0  # Use first column for labels
)

# Create DataLoader
train_loader = DataLoader(
    streaming_dataset, 
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print(f"Dataset created with input size: {streaming_dataset.input_size}")
print(f"Batch size: 32, Max samples: 100000")

## Step 6: Initialize and Train Model

Initialize the OCSVM model and train it on the training data.

**Available models**: LOF, OCSVM, VAE, Autoencoder, ICL, KitNET

In [ ]:
# Initialize model (change OCSVM to any other model: LOF, VAE, Autoencoder, ICL, KitNET)
ids = OCSVM(dataset_name=dataset_name, input_size=streaming_dataset.input_size, device=device)

# Train the model
ids.train_model(train_loader)

print("Model training complete!")

## Step 7: Save Trained Model

Save the trained model to disk for later use.

In [ ]:
# Save the trained model
ids.save()

print("Model saved successfully!")

## Step 8: Load Saved Model

Create a new model instance and load the previously saved weights.

In [ ]:
# Create new model instance
ids = OCSVM(dataset_name=dataset_name, input_size=streaming_dataset.input_size, device=device)

# Load the saved model
ids.load()

print("Model loaded successfully!")

## Step 9: Create Test DataLoader

Set up a test dataset for inference and evaluation.

In [ ]:
# Create test dataset (smaller sample)
test_dataset = StreamingCSVDataset(
    feature_csv_path=FEATURE_PATH,
    label_csv_path=LABEL_PATH,
    max_samples=5000,  # Smaller sample for testing
    label_column=0
)

# Create test DataLoader
test_loader = DataLoader(
    test_dataset, 
    batch_size=32, 
    shuffle=False, 
    num_workers=2
)

print(f"Test dataset created with {5000} samples")

## Step 10: Run Inference

Perform lightweight inference to get predictions without computing metrics.

In [ ]:
# Run inference
y_true, y_pred, reconstruction_errors = ids.infer(test_loader)

print(f"Inference complete!")
print(f"Predictions shape: {y_pred.shape}")
print(f"Number of anomalies detected: {y_pred.sum()}")
print(f"Anomaly rate: {y_pred.sum() / len(y_pred) * 100:.2f}%")

## Step 11: Full Evaluation

Perform comprehensive evaluation with metrics computation and plot generation.

In [ ]:
# Run full evaluation
ids.evaluate(test_loader)

print("\nEvaluation complete!")
print("Check the artifacts folder for:")
print("  - Confusion matrix plot")
print("  - ROC curve plot")
print("  - Anomaly score plot")
print("  - Metrics text file")

## Summary

This notebook demonstrated the complete Titli NIDS workflow:

✅ **Public API Methods Used:**
1. `train_model(train_loader)` - Train the model
2. `save(model_path)` - Save trained model to disk
3. `load(model_path)` - Load model from disk
4. `infer(test_loader)` - Lightweight inference (returns predictions)
5. `evaluate(test_loader)` - Full evaluation (computes metrics and generates plots)

**Available Models:**
- LOF (Local Outlier Factor)
- OCSVM (One-Class SVM)
- VAE (Variational Autoencoder)
- Autoencoder
- ICL (Instance-based Contrastive Learning)
- KitNET (Ensemble of Autoencoders)

All models follow the same consistent API!